In [3]:
import numpy  as np
import pandas as pd

from transformers import AutoTokenizer

token = 'USE_YOUR_OWN_KEY'
# first off, we read the dataset
dataset = pd.read_parquet("brabant-xvii.parquet").set_index(['split', 'project', 'file_id'])
llms    = pd.read_parquet('llm_data/llm_data.parquet')

pandas.core.series.Series

In [9]:
# then, for each model, we will compute the size in tokens of each text
def count_tokens(df, col_id, model):
    tokenizer   = AutoTokenizer.from_pretrained(model, token=token)
    token_count = lambda text: len(tokenizer(text, padding=False, truncation=False)['input_ids'])
    df[col_id]  = df['text'].map(token_count)
    return df

df = dataset
for llm in llms['key'].dropna().items():
    df = count_tokens(df, *llm)

df

tokenizer_config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/195k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/575k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/94.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

C:\ProgramData\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (2438 > 1024). Running this sequence through the model will result in indexing errors


... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/684k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1926 > 512). Running this sequence through the model will result in indexing errors


text  \
split project file_id                                                      
train pardons 299.txt  kaerle etc doen te wetene allen iegewoirdich e...   
              228.txt  remissie voer\ndiericken gheerits\nkaerle etc....   
              118.txt  philips etc. allen etc. saluit alsoe als tonse...   
              126.txt  philips byder gratien goidts coninck van casti...   
              373.txt  remissie van dootslage\nphilips byder gracien ...   
...                                                                  ...   
valid pardons 012.txt  maximilian par la grace de dieu esleu empereur...   
              099.txt  remissie\nalbert ende isabel clara eugenia etc...   
              315.txt  kaerle etc. doen te wetene allen iegewoirdigen...   
              300.txt  kaerle etc. doen te wetene allen iegewoirdige ...   
              004.txt  kaerle etc. ic doen te wetene allen iegenwoird...   

                       from-scratch  gpt2  gysbertv2  llama3.1-405b  \
split project file_id                                                 
train pardons 299.txt          1099  2438       1560           2157   
              228.txt          1366  3074       1998           2735   
              118.txt           402   756        537            661   
              126.txt          1147  2481       1531           2197   
              373.txt           934  1920       1331           1704   
...                             ...   ...        ...            ...   
valid pardons 012.txt          1131  2234       1903           1967   
              099.txt           426   974        591            853   
              315.txt          1435  3179       2075           2823   
              300.txt          1257  2830       1855           2510   
              004.txt          1873  4071       2669           3592   

                       llama3.1-70b  llama3.1-8b  mistral-7b  mistral-8x22b  \
split project file_id                                                         
train pardons 299.txt          2157         2157        2295           2295   
              228.txt          2735         2735        2910           2910   
              118.txt           661          661         728            728   
              126.txt          2197         2197        2333           2333   
              373.txt          1704         1704        1827           1827   
...                             ...          ...         ...            ...   
valid pardons 012.txt          1967         1967        2117           2117   
              099.txt           853          853         923            923   
              315.txt          2823         2823        3032           3032   
              300.txt          2510         2510        2677           2677   
              004.txt          3592         3592        3898           3898   

                       mistral-8x7b  mistral-large  mistral-nemo  xlm-roberta  
split project file_id                                                          
train pardons 299.txt          2295           2295          2081         1926  
              228.txt          2910           2910          2651         2395  
              118.txt           728            729           644          605  
              126.txt          2333           2334          2081         1932  
              373.txt          1827           1827          1642         1562  
...                             ...            ...           ...          ...  
valid pardons 012.txt          2117           2117          1828         1793  
              099.txt           923            923           841          741  
              315.txt          3032           3032          2719         2489  
              300.txt          2677           2677          2417         2219  
              004.txt          3898           3898          3466         3228  

[419 rows x 13 columns]

In [10]:
df.to_parquet('token_counts.parquet')

In [16]:
df.describe()

,from-scratch,gpt2,gysbertv2,llama3.1-405b,llama3.1-70b,llama3.1-8b,mistral-7b,mistral-8x22b,mistral-8x7b,mistral-large,mistral-nemo,xlm-roberta
count,419.000000,419.000000,419.000000,419.000000,419.000000,419.000000,419.000000,419.000000,419.000000,419.000000,419.000000,419.000000
mean,1360.281623,2910.730310,1940.906921,2577.260143,2577.260143,2577.260143,2764.541766,2764.541766,2764.541766,2764.725537,2471.754177,2285.398568
std,1718.513810,3695.506072,2421.688934,3333.873686,3333.873686,3333.873686,3551.454524,3551.454524,3551.454524,3551.479068,3196.653780,2917.380685
min,402.000000,756.000000,537.000000,661.000000,661.000000,661.000000,728.000000,728.000000,728.000000,729.000000,644.000000,605.000000
25%,1050.000000,2244.000000,1494.500000,1989.500000,1989.500000,1989.500000,2136.000000,2136.000000,2136.000000,2136.000000,1909.000000,1763.500000
50%,1213.000000,2587.000000,1741.000000,2304.000000,2304.000000,2304.000000,2476.000000,2476.000000,2476.000000,2476.000000,2208.000000,2053.000000
75%,1401.500000,3038.000000,2024.500000,2689.000000,2689.000000,2689.000000,2885.000000,2885.000000,2885.000000,2885.000000,2588.000000,2374.500000
max,33644.000000,73140.000000,47616.000000,65997.000000,65997.000000,65997.000000,70348.000000,70348.000000,70348.000000,70349.000000,63265.000000,57652.000000


In [20]:
all_but_text = [c for c in df.columns if c != 'text']
token_cmp = df[all_but_text].apply(lambda col: col / df['from-scratch'])
token_cmp

from-scratch      gpt2  gysbertv2  llama3.1-405b  \
split project file_id                                                     
train pardons 299.txt           1.0  2.218380   1.419472       1.962693   
              228.txt           1.0  2.250366   1.462665       2.002196   
              118.txt           1.0  1.880597   1.335821       1.644279   
              126.txt           1.0  2.163034   1.334786       1.915432   
              373.txt           1.0  2.055675   1.425054       1.824411   
...                             ...       ...        ...            ...   
valid pardons 012.txt           1.0  1.975243   1.682582       1.739169   
              099.txt           1.0  2.286385   1.387324       2.002347   
              315.txt           1.0  2.215331   1.445993       1.967247   
              300.txt           1.0  2.251392   1.475736       1.996818   
              004.txt           1.0  2.173518   1.424987       1.917779   

                       llama3.1-70b  llama3.1-8b  mistral-7b  mistral-8x22b  \
split project file_id                                                         
train pardons 299.txt      1.962693     1.962693    2.088262       2.088262   
              228.txt      2.002196     2.002196    2.130307       2.130307   
              118.txt      1.644279     1.644279    1.810945       1.810945   
              126.txt      1.915432     1.915432    2.034002       2.034002   
              373.txt      1.824411     1.824411    1.956103       1.956103   
...                             ...          ...         ...            ...   
valid pardons 012.txt      1.739169     1.739169    1.871795       1.871795   
              099.txt      2.002347     2.002347    2.166667       2.166667   
              315.txt      1.967247     1.967247    2.112892       2.112892   
              300.txt      1.996818     1.996818    2.129674       2.129674   
              004.txt      1.917779     1.917779    2.081153       2.081153   

                       mistral-8x7b  mistral-large  mistral-nemo  xlm-roberta  
split project file_id                                                          
train pardons 299.txt      2.088262       2.088262      1.893540     1.752502  
              228.txt      2.130307       2.130307      1.940703     1.753294  
              118.txt      1.810945       1.813433      1.601990     1.504975  
              126.txt      2.034002       2.034874      1.814298     1.684394  
              373.txt      1.956103       1.956103      1.758030     1.672377  
...                             ...            ...           ...          ...  
valid pardons 012.txt      1.871795       1.871795      1.616269     1.585323  
              099.txt      2.166667       2.166667      1.974178     1.739437  
              315.txt      2.112892       2.112892      1.894774     1.734495  
              300.txt      2.129674       2.129674      1.922832     1.765314  
              004.txt      2.081153       2.081153      1.850507     1.723438  

[419 rows x 12 columns]

In [23]:
token_cmp.mean()

from-scratch     1.000000
gpt2             2.146181
gysbertv2        1.430286
llama3.1-405b    1.897392
llama3.1-70b     1.897392
llama3.1-8b      1.897392
mistral-7b       2.036256
mistral-8x22b    2.036256
mistral-8x7b     2.036256
mistral-large    2.036431
mistral-nemo     1.819814
xlm-roberta      1.682857
dtype: float64